# Bulk Loading And Export

## Quick notes
### What this covers
- Getting lots of rows IN: CSV, JSON, `INSERT ... SELECT`, `executemany`, staging tables, upserts, deduplication and handling bad rows.
- Making loads fast and safe: transactions, batch size, a few PRAGMAs, building indexes after the load.
- Getting rows OUT: CSV and JSON export, generating test data with recursive CTEs, the `sqlite3` command-line dot-commands, large-object notes.
### Key points
- One transaction around many inserts is the single biggest speed-up: each commit forces a disk sync, so fewer commits = far less waiting.
- Load into a loose STAGING table first, validate with SQL, then merge only the good rows into the real table.
- `INSERT ... ON CONFLICT DO UPDATE` makes a reload idempotent: run it twice and the second run changes nothing.
- Collect rejected rows (with the reason) instead of aborting the whole load or silently dropping them.
- `json_each` turns a JSON array into rows; `json_group_array` / `json_object` turn rows into JSON.
- Recursive CTEs plus `printf` generate readable test data with no external tool.
### Gotchas (the part worth revising twice)
- Autocommit (no `BEGIN`) means one commit per statement. 100,000 single inserts = 100,000 commits.
- One giant transaction is all-or-nothing: one bad row at the end undoes everything. Batches keep earlier progress.
- `ON CONFLICT` with duplicates inside the SAME incoming batch: rows apply one after another, so the order decides the winner. Deduplicate first.
- `INSERT ... SELECT ... ON CONFLICT` needs a `WHERE true` (or any WHERE) to avoid a parser ambiguity.
- `synchronous = OFF` and `journal_mode = OFF` are fast and unsafe: a crash can corrupt the file. Use only for data you can rebuild.
- CSV is text: convert types yourself (`"4.50"` to cents), and let Python's `csv` module handle quotes and embedded commas.
- `length()` counts characters for text but bytes for BLOBs.
### When it matters
Data imports, nightly feeds, migrations, test fixtures, report exports, and any time a loop of single `INSERT`s feels slow.
### One-line recap example
```sql
BEGIN;
INSERT INTO products SELECT sku, name, price_cents, category FROM staging WHERE problem IS NULL ON CONFLICT(sku) DO UPDATE SET price_cents = excluded.price_cents;
COMMIT;
```

## 01. Seed data
- A small product catalogue: `categories` (allowed values) and `products` with a CHECK and a foreign key, so bad loads get caught by the database.
- `load_test` is a scratch table for the commit-counting demos.
- Foreign keys are ON in this notebook (see 05-DDL-And-Integrity/01-Constraints-And-Keys).

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE categories (name TEXT PRIMARY KEY);
INSERT INTO categories VALUES ('paper'), ('office'), ('art');
CREATE TABLE products (
    sku         TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    price_cents INTEGER NOT NULL CHECK (price_cents >= 0),
    category    TEXT NOT NULL REFERENCES categories(name)
);
CREATE TABLE load_test (id INTEGER PRIMARY KEY, v INTEGER);
SELECT name FROM categories ORDER BY name;

name  
------
art   
office
paper 
(3 rows)



## 02. Loading CSV text with Python, then executemany
- SQLite itself has no `LOAD CSV` statement. Parse the CSV in Python (the `csv` module handles quotes and commas inside fields), then insert in one go with `executemany`.
- Convert types while parsing: a price like `4.50` becomes `450` cents (use `Decimal`, not float, for money).
- Wrap the whole `executemany` in one transaction. Here the data is an inline string; for a file you would open it with `newline=''`.
- The row `B-201` has a comma inside the name: it arrives intact.

In [3]:
import csv, io
from decimal import Decimal

csv_text = '''sku,name,price,category
A-100,Notebook,4.50,paper
A-101,Pen,1.20,paper
B-200,Stapler,8.00,office
B-201,"Tape, clear",2.75,office
'''
rows = [(r["sku"], r["name"], int(Decimal(r["price"]) * 100), r["category"])
        for r in csv.DictReader(io.StringIO(csv_text))]
con.execute("BEGIN")
con.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", rows)
con.execute("COMMIT")
print("rows parsed:", len(rows))
print("row 4 kept its comma:", rows[3])

rows parsed: 4
row 4 kept its comma: ('B-201', 'Tape, clear', 275, 'office')


In [4]:
%%sql
SELECT sku, name, price_cents, category FROM products ORDER BY sku;

sku   | name        | price_cents | category
------+-------------+-------------+---------
A-100 | Notebook    | 450         | paper   
A-101 | Pen         | 120         | paper   
B-200 | Stapler     | 800         | office  
B-201 | Tape, clear | 275         | office  
(4 rows)



## 03. INSERT ... SELECT and multi-row VALUES
- `INSERT INTO target (cols) SELECT ... FROM source` copies and reshapes rows entirely inside the database. No Python round trip, so it is the fastest bulk path.
- Always list the target columns. Positional inserts break the day someone adds a column.
- A multi-row `VALUES (...), (...)` is one statement, so one implicit transaction.
- `SELECT changes()` right after tells you how many rows the statement touched.

In [5]:
%%sql
CREATE TABLE products_paper (sku TEXT PRIMARY KEY, name TEXT NOT NULL, price_dollars REAL NOT NULL);
INSERT INTO products_paper (sku, name, price_dollars)
SELECT sku, upper(name), price_cents / 100.0 FROM products WHERE category = 'paper' ORDER BY sku;
SELECT changes() AS rows_copied;
INSERT INTO products_paper VALUES ('Z-1', 'EXTRA A', 1.0), ('Z-2', 'EXTRA B', 2.0), ('Z-3', 'EXTRA C', 3.0);
SELECT changes() AS rows_inserted_by_values;
SELECT sku, name, price_dollars FROM products_paper ORDER BY sku;

rows_copied
-----------
2          
(1 row)

rows_inserted_by_values
-----------------------
3                      
(1 row)

sku   | name     | price_dollars
------+----------+--------------
A-100 | NOTEBOOK | 4.5          
A-101 | PEN      | 1.2          
Z-1   | EXTRA A  | 1.0          
Z-2   | EXTRA B  | 2.0          
Z-3   | EXTRA C  | 3.0          
(5 rows)



## 04. Loading JSON with json_each
- `json_each(text)` returns one row per array element (or object member). Pull fields out with `json_extract`.
- Good for API payloads and exports from other systems: pass the whole JSON text as ONE parameter and let SQL do the splitting.
- Missing keys come out as NULL, so a NOT NULL column rejects the row. Check shape with staging if you do not trust the source.
- Order of rows follows the array order.

In [6]:
%%sql
INSERT INTO products (sku, name, price_cents, category)
SELECT json_extract(value, '$.sku'),
       json_extract(value, '$.name'),
       json_extract(value, '$.price_cents'),
       json_extract(value, '$.category')
FROM json_each('[
  {"sku":"J-900","name":"Sketchbook","price_cents":1250,"category":"art"},
  {"sku":"J-901","name":"Charcoal","price_cents":640,"category":"art"}
]');
SELECT changes() AS rows_loaded;
SELECT sku, name, price_cents FROM products WHERE sku LIKE 'J-%' ORDER BY sku;
SELECT key, value FROM json_each('{"a": 1, "b": [2, 3]}') ORDER BY key;

rows_loaded
-----------
2          
(1 row)

sku   | name       | price_cents
------+------------+------------
J-900 | Sketchbook | 1250       
J-901 | Charcoal   | 640        
(2 rows)

key | value
----+------
a   | 1    
b   | [2,3]
(2 rows)



## 05. Staging table, validate, then merge
- The safest pattern for messy files: load EVERYTHING as text into a staging table with no constraints, check it with SQL, merge only the good rows.
- Nothing can fail during the raw load, so you always get a complete picture of the file, including the bad lines.
- A view `v_staging_check` gives each row a `problem` (NULL when fine). The first matching rule wins.
- `GLOB '*[^0-9.]*'` finds any character that is not a digit or dot: a cheap "is this a number" test.
- Bad vs good: inserting the file straight into `products` would reject only two of the five bad rows (the negative price and the unknown category) and would stop at the first one. The empty sku, the empty name and `abc` as a price would be stored silently, because the table has no rule against them. Staging reports all five problems at once.

In [7]:
staging_csv = '''sku,name,price,category
C-300,Marker,3.10,office
C-301,Folder,0.90,paper
,Ghost,1.00,paper
C-303,Ruler,abc,office
C-304,Glue,-2.00,office
C-305,,1.00,office
C-306,Clip,0.15,hardware
C-307,Paint,5.50,art
'''
con.execute("CREATE TABLE staging_products (line_no INTEGER, sku TEXT, name TEXT, price TEXT, category TEXT)")
con.execute("BEGIN")
con.executemany("INSERT INTO staging_products VALUES (?, ?, ?, ?, ?)",
    [(i, r["sku"], r["name"], r["price"], r["category"])
     for i, r in enumerate(csv.DictReader(io.StringIO(staging_csv)), start=2)])
con.execute("COMMIT")
print("raw rows staged:", con.execute("SELECT count(*) FROM staging_products").fetchone()[0])

raw rows staged: 8


In [8]:
%%sql
CREATE VIEW v_staging_check AS
SELECT line_no, sku, name, price, category,
       CASE WHEN trim(sku) = ''  THEN 'missing sku'
            WHEN trim(name) = '' THEN 'missing name'
            WHEN price = '' OR price GLOB '*[^0-9.]*' OR price NOT GLOB '[0-9]*' THEN 'bad price'
            WHEN category NOT IN (SELECT name FROM categories) THEN 'unknown category'
       END AS problem
FROM staging_products;
SELECT line_no, sku, price, category, problem FROM v_staging_check WHERE problem IS NOT NULL ORDER BY line_no;
INSERT INTO products (sku, name, price_cents, category)
SELECT sku, name, CAST(round(CAST(price AS REAL) * 100) AS INTEGER), category
FROM v_staging_check WHERE problem IS NULL ORDER BY line_no;
SELECT changes() AS good_rows_merged;
SELECT sku, name, price_cents FROM products WHERE sku LIKE 'C-%' ORDER BY sku;

line_no | sku   | price | category | problem         
--------+-------+-------+----------+-----------------
4       |       | 1.00  | paper    | missing sku     
5       | C-303 | abc   | office   | bad price       
6       | C-304 | -2.00 | office   | bad price       
7       | C-305 | 1.00  | office   | missing name    
8       | C-306 | 0.15  | hardware | unknown category
(5 rows)

good_rows_merged
----------------
3               
(1 row)

sku   | name   | price_cents
------+--------+------------
C-300 | Marker | 310        
C-301 | Folder | 90         
C-307 | Paint  | 550        
(3 rows)



## 06. Autocommit vs one transaction: count the commits
- Without `BEGIN`, every statement is its own transaction, and every transaction ends with a commit that waits for the disk. That wait, not the insert itself, is the cost.
- We cannot print time here (it would change every run), so we count what causes it: commits.
- The helper counts a commit each time a statement leaves no transaction open (autocommit), or each explicit `COMMIT`.
- Same 200 rows each time. Only the number of commits changes.
- See also 06-Data-Modification/02-Transactions.

In [9]:
def load_rows(n_rows, batch):
    """Load n_rows into load_test; batch=1 means plain autocommit (no BEGIN). Returns commits."""
    con.execute("DELETE FROM load_test")
    rows = [(i, i * 2) for i in range(1, n_rows + 1)]
    commits = 0
    if batch == 1:
        for r in rows:
            con.execute("INSERT INTO load_test VALUES (?, ?)", r)
            commits += not con.in_transaction          # autocommit: this statement committed itself
    else:
        for i in range(0, n_rows, batch):
            con.execute("BEGIN")
            con.executemany("INSERT INTO load_test VALUES (?, ?)", rows[i:i + batch])
            con.execute("COMMIT")
            commits += 1
    return commits, con.execute("SELECT count(*) FROM load_test").fetchone()[0]

print("batch size | commits | rows loaded")
for b in (1, 10, 50, 200):
    c, n = load_rows(200, b)
    print(f"{b:>10} | {c:>7} | {n}")

batch size | commits | rows loaded
         1 |     200 | 200
        10 |      20 | 200
        50 |       4 | 200
       200 |       1 | 200


## 07. Batch size: progress vs all-or-nothing
- One huge transaction: fewest commits, but a single bad row rolls back everything, and the write lock is held for the whole load.
- Batches of N rows: a few more commits, but earlier batches stay committed and other writers can slip in between.
- Rule of thumb: a few thousand rows per batch is plenty; going bigger gains almost nothing.
- Demo: row 120 is a duplicate id. In one transaction nothing is kept (0 rows). In batches of 50 the first two batches (100 rows) survive.

In [10]:
def load_with_bad_row(batch):
    con.execute("DELETE FROM load_test")
    rows = [(i, i) for i in range(1, 201)] ; rows[119] = (119, 0)   # id 119 appears twice
    committed = 0
    for i in range(0, len(rows), batch):
        con.execute("BEGIN")
        try:
            con.executemany("INSERT INTO load_test VALUES (?, ?)", rows[i:i + batch])
            con.execute("COMMIT"); committed += 1
        except sqlite3.IntegrityError:
            con.execute("ROLLBACK")
            break
    return committed, con.execute("SELECT count(*) FROM load_test").fetchone()[0]

print("one transaction  -> batches committed, rows kept:", load_with_bad_row(200))
print("batches of 50    -> batches committed, rows kept:", load_with_bad_row(50))
_ = con.execute("DELETE FROM load_test")

one transaction  -> batches committed, rows kept: (0, 0)
batches of 50    -> batches committed, rows kept: (2, 100)


## 08. ON CONFLICT merge loads (upsert)
- `ON CONFLICT(key) DO UPDATE SET ...` inserts new rows and updates existing ones in one statement. `excluded.col` is the incoming value.
- Add a `WHERE` to the `DO UPDATE` so unchanged rows are skipped. Then `changes()` counts only real changes, and no needless writes happen.
- The incoming batch has: a price change (A-100), an unchanged row (A-101), a renamed row (B-200) and a new row (D-400). Expected `changes()` = 3.
- Run it again: nothing differs any more, so 0 changes. That is what idempotent means.
- `WHERE true` after the `FROM` avoids the SELECT-versus-ON-CONFLICT parsing ambiguity.

In [11]:
%%sql
CREATE TABLE incoming (sku TEXT, name TEXT, price_cents INTEGER, category TEXT);
INSERT INTO incoming VALUES
    ('A-100', 'Notebook', 475, 'paper'), ('A-101', 'Pen', 120, 'paper'),
    ('B-200', 'Heavy Stapler', 800, 'office'), ('D-400', 'Brush', 650, 'art');
INSERT INTO products (sku, name, price_cents, category)
SELECT sku, name, price_cents, category FROM incoming WHERE true
ON CONFLICT(sku) DO UPDATE SET
    name = excluded.name, price_cents = excluded.price_cents, category = excluded.category
WHERE products.name IS NOT excluded.name
   OR products.price_cents IS NOT excluded.price_cents
   OR products.category IS NOT excluded.category;
SELECT changes() AS first_run_changes;
INSERT INTO products (sku, name, price_cents, category)
SELECT sku, name, price_cents, category FROM incoming WHERE true
ON CONFLICT(sku) DO UPDATE SET
    name = excluded.name, price_cents = excluded.price_cents, category = excluded.category
WHERE products.name IS NOT excluded.name
   OR products.price_cents IS NOT excluded.price_cents
   OR products.category IS NOT excluded.category;
SELECT changes() AS second_run_changes;
SELECT sku, name, price_cents FROM products WHERE sku IN ('A-100','A-101','B-200','D-400') ORDER BY sku;

first_run_changes
-----------------
3                
(1 row)

second_run_changes
------------------
0                 
(1 row)

sku   | name          | price_cents
------+---------------+------------
A-100 | Notebook      | 475        
A-101 | Pen           | 120        
B-200 | Heavy Stapler | 800        
D-400 | Brush         | 650        
(4 rows)



## 09. Deduplicating on load
- Real feeds repeat keys. Inserting duplicates in one statement fails the WHOLE statement on the first conflict (nothing is inserted).
- Decide the rule first: first row wins, last row wins, or merge. Then implement it explicitly.
- "Last row wins" with a window function: `row_number() OVER (PARTITION BY sku ORDER BY line_no DESC) = 1` (see 03-Window-Functions/01-Window-Functions).
- "First row wins" is `INSERT OR IGNORE` with an `ORDER BY line_no`, or the same window with `ASC`.
- Below, E-500 appears three times (lines 1, 3, 4). The last one (line 4, `Easel Pro`) must win.

In [12]:
%%sql
CREATE TABLE staging_dupes (line_no INTEGER, sku TEXT, name TEXT, price_cents INTEGER, category TEXT);
INSERT INTO staging_dupes VALUES
    (1, 'E-500', 'Easel', 9000, 'art'), (2, 'E-501', 'Canvas', 1500, 'art'),
    (3, 'E-500', 'Easel', 9500, 'art'), (4, 'E-500', 'Easel Pro', 9900, 'art');

In [13]:
%%sql expect_error
INSERT INTO products SELECT sku, name, price_cents, category FROM staging_dupes ORDER BY line_no;
SELECT count(*) AS rows_after_failed_insert FROM products WHERE sku LIKE 'E-%';

ERROR: UNIQUE constraint failed: products.sku

rows_after_failed_insert
------------------------
0                       
(1 row)



In [14]:
%%sql
INSERT INTO products (sku, name, price_cents, category)
SELECT sku, name, price_cents, category FROM (
    SELECT *, row_number() OVER (PARTITION BY sku ORDER BY line_no DESC) AS rn FROM staging_dupes
) WHERE rn = 1 ORDER BY sku;
SELECT changes() AS rows_loaded;
SELECT sku, name, price_cents FROM products WHERE sku LIKE 'E-%' ORDER BY sku;

rows_loaded
-----------
2          
(1 row)

sku   | name      | price_cents
------+-----------+------------
E-500 | Easel Pro | 9900       
E-501 | Canvas    | 1500       
(2 rows)



## 10. Handling bad rows: collect rejects
- Do not abort the whole load for one bad line, and do not drop bad lines silently. Keep them in a `load_rejects` table with the line number and the reason.
- Two kinds of failure: Python-side checks (wrong column count, price not a number) and database-side constraints (CHECK, UNIQUE, FOREIGN KEY).
- Inside one transaction, a failed single-row `INSERT` undoes only that statement, so the loop can carry on and commit the good rows.
- The reject reason is the database's own error text. Re-run after the supplier fixes the file.

In [15]:
con.execute("CREATE TABLE load_rejects (line_no INTEGER, raw TEXT, reason TEXT)")
feed = '''sku,name,price,category
F-1,Compass,3.20,office
F-2,Stencil,-1.00,art
F-3,Eraser,0.50
F-4,Pastel,7.00,pottery
A-100,Notebook,4.50,paper
F-6,Easel stand,twelve,art
F-7,Sticker,0.25,paper
'''
good = 0
con.execute("BEGIN")
for line_no, line in enumerate(feed.splitlines()[1:], start=2):
    fields = next(csv.reader([line]))
    try:
        if len(fields) != 4:
            raise ValueError(f"expected 4 fields, got {len(fields)}")
        sku, name, price, cat = fields
        try:
            cents = int(Decimal(price) * 100)
        except ArithmeticError:
            raise ValueError(f"price {price!r} is not a number")
        con.execute("INSERT INTO products VALUES (?, ?, ?, ?)", (sku, name, cents, cat))
        good += 1
    except (ValueError, sqlite3.IntegrityError) as exc:
        con.execute("INSERT INTO load_rejects VALUES (?, ?, ?)", (line_no, line, f"{type(exc).__name__}: {exc}"))
con.execute("COMMIT")
print("good rows:", good, "| rejected:", con.execute("SELECT count(*) FROM load_rejects").fetchone()[0])

good rows: 2 | rejected: 5


In [16]:
%%sql
SELECT line_no, raw, reason FROM load_rejects ORDER BY line_no;

line_no | raw                        | reason                                                   
--------+----------------------------+----------------------------------------------------------
3       | F-2,Stencil,-1.00,art      | IntegrityError: CHECK constraint failed: price_cents >= 0
4       | F-3,Eraser,0.50            | ValueError: expected 4 fields, got 3                     
5       | F-4,Pastel,7.00,pottery    | IntegrityError: FOREIGN KEY constraint failed            
6       | A-100,Notebook,4.50,paper  | IntegrityError: UNIQUE constraint failed: products.sku   
7       | F-6,Easel stand,twelve,art | ValueError: price 'twelve' is not a number               
(5 rows)



## 11. PRAGMAs for bulk loads: journal_mode and synchronous
- `journal_mode`: how SQLite keeps a transaction undoable. `DELETE` (default) uses a rollback-journal file; `WAL` appends to a write-ahead log (readers and the writer do not block each other); `MEMORY` / `OFF` keep the journal in RAM or drop it.
- `synchronous`: how hard SQLite waits for the disk. `2` FULL (safest), `1` NORMAL (fast and still safe in WAL mode, may lose the last commits on power loss, never corrupts), `0` OFF (no waiting: an OS crash can corrupt the file).
- Safe fast recipe for a rebuildable import: build into a NEW file with `synchronous = OFF`, check it, then rename it over the old one.
- Other helpers: `PRAGMA foreign_keys = OFF` during the load plus `PRAGMA foreign_key_check` afterwards; a bigger `cache_size`; `temp_store = MEMORY`.
- File-backed demo below uses a temporary file and removes it. The notebook's own in-memory database always reports `memory`.

In [17]:
import tempfile, os, shutil

print("in-memory journal_mode:", con.execute("PRAGMA journal_mode").fetchone()[0])
tmpdir = tempfile.mkdtemp()
try:
    fdb = sqlite3.connect(os.path.join(tmpdir, "bulk.db"), isolation_level=None)
    print("file db default journal_mode:", fdb.execute("PRAGMA journal_mode").fetchone()[0])
    print("file db default synchronous: ", fdb.execute("PRAGMA synchronous").fetchone()[0], "(2 = FULL)")
    print("switch to WAL ->", fdb.execute("PRAGMA journal_mode = WAL").fetchone()[0])
    fdb.execute("PRAGMA synchronous = NORMAL")
    print("synchronous now:", fdb.execute("PRAGMA synchronous").fetchone()[0], "(1 = NORMAL)")
    fdb.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, v INTEGER)")
    fdb.execute("BEGIN")
    fdb.executemany("INSERT INTO t VALUES (?, ?)", [(i, i) for i in range(1, 501)])
    fdb.execute("COMMIT")
    print("rows loaded in WAL mode:", fdb.execute("SELECT count(*) FROM t").fetchone()[0])
    fdb.close()
finally:
    shutil.rmtree(tmpdir)
print("temp folder removed:", not os.path.exists(tmpdir))

in-memory journal_mode: memory
file db default journal_mode: delete
file db default synchronous:  2 (2 = FULL)
switch to WAL -> wal
synchronous now: 1 (1 = NORMAL)
rows loaded in WAL mode: 500
temp folder removed: True


## 12. Build indexes after the load
- Every index is another b-tree that must be updated for every inserted row. Loading into a table that already has five indexes does six writes per row.
- For a big initial load: create the table, load the rows, THEN `CREATE INDEX`. SQLite builds an index from sorted data in one pass, which is much cheaper.
- For incremental loads into a live table you usually keep the indexes. The trade-off is yours.
- The query plan proves the index is in place afterwards (see 07-Performance for plan reading).

In [18]:
%%sql
CREATE TABLE clicks (id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL, url TEXT NOT NULL);
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 500)
INSERT INTO clicks SELECT i, i % 25, printf('/page/%03d', i % 40) FROM n;
SELECT count(*) AS clicks_loaded FROM clicks;
EXPLAIN QUERY PLAN SELECT count(*) FROM clicks WHERE user_id = 7;
CREATE INDEX idx_clicks_user ON clicks(user_id);
EXPLAIN QUERY PLAN SELECT count(*) FROM clicks WHERE user_id = 7;

clicks_loaded
-------------
500          
(1 row)

id | parent | notused | detail     
---+--------+---------+------------
3  | 0      | 216     | SCAN clicks
(1 row)

id | parent | notused | detail                                                        
---+--------+---------+---------------------------------------------------------------
3  | 0      | 53      | SEARCH clicks USING COVERING INDEX idx_clicks_user (user_id=?)
(1 row)



## 13. Exporting to CSV from Python
- Run a `SELECT`, write rows with `csv.writer`. It quotes fields that contain commas, quotes or newlines, so the file round-trips.
- Take the header from `cursor.description`. Format numbers in SQL (`printf('%.2f', ...)`) so the CSV has fixed decimals.
- Always `ORDER BY`: an export with no order changes between runs and makes diffs useless.
- Here the text goes to a string buffer; for a file you would `open(name, 'w', newline='')`.

In [19]:
buf = io.StringIO()
w = csv.writer(buf, lineterminator="\n")
cur = con.execute("""SELECT sku, name, printf('%.2f', price_cents / 100.0) AS price, category
                     FROM products WHERE sku IN ('A-100','B-200','B-201','J-900') ORDER BY sku""")
w.writerow([d[0] for d in cur.description])
w.writerows(cur)
print(buf.getvalue())

sku,name,price,category
A-100,Notebook,4.75,paper
B-200,Heavy Stapler,8.00,office
B-201,"Tape, clear",2.75,office
J-900,Sketchbook,12.50,art



## 14. Exporting to JSON in SQL: json_group_array and json_object
- `json_object('key', value, ...)` builds one JSON object per row. `json_group_array(x)` collects many rows into a JSON array.
- In SQLite 3.44+ the aggregate accepts its own `ORDER BY`: `json_group_array(x ORDER BY sku)`. That keeps array order deterministic.
- Nest by grouping: one object per category, each holding an array of SKUs. Wrap an inner JSON string in `json(...)` so it nests as JSON, not as a quoted string.
- Python then prints it nicely. JSON Lines (one object per line) is just one `json_object` per row.

In [20]:
%%sql
SELECT json_group_array(json_object('sku', sku, 'price', price_cents / 100.0) ORDER BY sku) AS products_json
FROM products WHERE sku IN ('A-100','A-101','J-900');
SELECT json_group_array(json_object('category', category, 'skus', json(skus)) ORDER BY category) AS by_category
FROM (SELECT category, json_group_array(sku ORDER BY sku) AS skus
      FROM products WHERE sku IN ('A-100','A-101','B-200','J-900') GROUP BY category);

products_json                                                                          
---------------------------------------------------------------------------------------
[{"sku":"A-100","price":4.75},{"sku":"A-101","price":1.2},{"sku":"J-900","price":12.5}]
(1 row)

by_category                                                                                                               
--------------------------------------------------------------------------------------------------------------------------
[{"category":"art","skus":["J-900"]},{"category":"office","skus":["B-200"]},{"category":"paper","skus":["A-100","A-101"]}]
(1 row)



In [21]:
import json
js = con.execute("""SELECT json_group_array(json_object('category', category, 'skus', json(skus)) ORDER BY category)
                    FROM (SELECT category, json_group_array(sku ORDER BY sku) AS skus
                          FROM products WHERE sku IN ('A-100','A-101','B-200','J-900') GROUP BY category)""").fetchone()[0]
print(json.dumps(json.loads(js), indent=2))
print("JSON Lines:")
for (line,) in con.execute("SELECT json_object('sku', sku, 'name', name) FROM products WHERE sku LIKE 'A-%' ORDER BY sku"):
    print(line)

[
  {
    "category": "art",
    "skus": [
      "J-900"
    ]
  },
  {
    "category": "office",
    "skus": [
      "B-200"
    ]
  },
  {
    "category": "paper",
    "skus": [
      "A-100",
      "A-101"
    ]
  }
]
JSON Lines:
{"sku":"A-100","name":"Notebook"}
{"sku":"A-101","name":"Pen"}


## 15. Generating test data with recursive CTEs and printf
- A recursive CTE that counts 1..N is a row factory. `printf` turns the counter into padded, readable values (`SKU-0007`).
- Use arithmetic on the counter for variety: `i % 3` for categories, `(i * 7919) % 900` for scattered prices. The result is repeatable, unlike `random()`.
- Use `date('2026-01-01', '+' || (i * 37 % 365) || ' days')` for spread-out dates from a fixed anchor.
- Generate once into a scratch table, then check with aggregates that the shape is what you wanted.

In [22]:
%%sql
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 8)
SELECT printf('SKU-%04d', i) AS sku,
       printf('user%05d@example.com', i) AS email,
       100 + (i * 7919) % 900 AS price_cents,
       CASE i % 3 WHEN 0 THEN 'paper' WHEN 1 THEN 'office' ELSE 'art' END AS category,
       date('2026-01-01', '+' || (i * 37 % 365) || ' days') AS added_on
FROM n ORDER BY i;
CREATE TABLE gen_items (sku TEXT PRIMARY KEY, price_cents INTEGER, category TEXT);
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 1000)
INSERT INTO gen_items SELECT printf('SKU-%04d', i), 100 + (i * 7919) % 900,
       CASE i % 3 WHEN 0 THEN 'paper' WHEN 1 THEN 'office' ELSE 'art' END FROM n;
SELECT count(*) AS n, min(price_cents) AS lowest, max(price_cents) AS highest,
       count(DISTINCT category) AS categories FROM gen_items;

sku      | email                 | price_cents | category | added_on  
---------+-----------------------+-------------+----------+-----------
SKU-0001 | user00001@example.com | 819         | office   | 2026-02-07
SKU-0002 | user00002@example.com | 638         | art      | 2026-03-16
SKU-0003 | user00003@example.com | 457         | paper    | 2026-04-22
SKU-0004 | user00004@example.com | 276         | office   | 2026-05-29
SKU-0005 | user00005@example.com | 995         | art      | 2026-07-05
SKU-0006 | user00006@example.com | 814         | paper    | 2026-08-11
SKU-0007 | user00007@example.com | 633         | office   | 2026-09-17
SKU-0008 | user00008@example.com | 452         | art      | 2026-10-24
(8 rows)

n    | lowest | highest | categories
-----+--------+---------+-----------
1000 | 100    | 999     | 3         
(1 row)



## 16. The sqlite3 CLI: .import, .mode, .output (not executed here)
- The command-line shell has dot-commands that do bulk work without any Python. These are shell commands, not SQL, so the code blocks below are for reading only.
- Import a CSV (first line is the header, skipped with `--skip 1` when the table already exists):
```text
sqlite3 shop.db
.mode csv
.import --skip 1 products.csv staging_products
```
- If the table does not exist, `.import` creates it from the header line with every column as TEXT. That is why staging tables fit so well.
- Export a query as CSV with a header, to a file:
```text
.headers on
.mode csv
.output products_export.csv
SELECT sku, name, price_cents FROM products ORDER BY sku;
.output stdout
```
- `.once FILE` sends only the NEXT result to a file. `.mode json` prints rows as JSON, `.mode table` draws a boxed table, `.mode line` prints one column per line.
- Whole-database tools: `.dump` writes the schema and data as SQL text, `.read script.sql` runs a script, `.schema products` shows the CREATE text, `.tables` lists tables, `.quit` exits.
- Run a script non-interactively: `sqlite3 shop.db < load.sql` or `sqlite3 shop.db ".import ..."`.
- Gotcha: `.import` into an existing table loads the header as a data row unless you add `--skip 1`.

## 17. Large objects: BLOBs and big text
- A BLOB column stores raw bytes. `zeroblob(n)` reserves n zero bytes cheaply, and Python's `connection.blobopen(...)` reads or writes a slice without loading the whole value.
- `length(text)` counts characters, `length(blob)` counts bytes. UTF-8 `é` is one character but two bytes.
- The default maximum size of one value is 1 billion bytes. Practical advice: keep files above a few MB on disk and store only the path plus a hash in the database; keep small things (icons, thumbnails) as BLOBs.
- Put big BLOB columns in their own table, so scanning the main table does not drag the payload through memory.
- Reclaim space after deleting big values with `VACUUM`; a delete alone only frees pages inside the file.

In [23]:
%%sql
CREATE TABLE files (id INTEGER PRIMARY KEY, name TEXT NOT NULL, data BLOB NOT NULL);
INSERT INTO files VALUES (1, 'zeros.bin', zeroblob(1000));
SELECT id, name, typeof(data) AS type, length(data) AS bytes FROM files ORDER BY id;
SELECT length('héllo') AS chars, length(CAST('héllo' AS BLOB)) AS bytes;

id | name      | type | bytes
---+-----------+------+------
1  | zeros.bin | blob | 1000 
(1 row)

chars | bytes
------+------
5     | 6    
(1 row)



In [24]:
if hasattr(sqlite3.Connection, 'blobopen'):
    with con.blobopen("files", "data", 1) as b:
        b.write(b"HEADER")
        b.seek(990)
        b.write(b"TRAILER!!!")
        print("blob size seen by blobopen:", len(b))
        b.seek(0)
        print("first 6 bytes:", b.read(6))
    print("hex of the first 8 bytes via SQL:", con.execute("SELECT hex(substr(data, 1, 8)) FROM files WHERE id = 1").fetchone()[0])
else:
    print('SKIP: incremental BLOB I/O requires Python 3.11+')

blob size seen by blobopen: 1000
first 6 bytes: b'HEADER'
hex of the first 8 bytes via SQL: 4845414445520000


## 18. Reconcile after a load
- A load is not finished until you compare the numbers: rows read vs rows loaded vs rows rejected, plus a control total (here the sum of prices).
- If `read = loaded + rejected + skipped_as_duplicates` does not hold, a row went missing. Investigate before trusting the data.
- Compare SQL aggregates between staging and target for the rows that were merged.
- Keep the staging table (or a log row) until the checks pass, then drop it.

In [25]:
%%sql
SELECT (SELECT count(*) FROM staging_products) AS read_rows,
       (SELECT count(*) FROM v_staging_check WHERE problem IS NULL) AS good_rows,
       (SELECT count(*) FROM v_staging_check WHERE problem IS NOT NULL) AS rejected_rows,
       (SELECT count(*) FROM products WHERE sku IN (SELECT sku FROM v_staging_check WHERE problem IS NULL)) AS found_in_products;
SELECT (SELECT sum(CAST(round(CAST(price AS REAL) * 100) AS INTEGER)) FROM v_staging_check WHERE problem IS NULL) AS staging_total_cents,
       (SELECT sum(price_cents) FROM products WHERE sku IN (SELECT sku FROM v_staging_check WHERE problem IS NULL)) AS products_total_cents;
DROP VIEW v_staging_check;
DROP TABLE staging_products;
SELECT count(*) AS products_total FROM products;

read_rows | good_rows | rejected_rows | found_in_products
----------+-----------+---------------+------------------
8         | 3         | 5             | 3                
(1 row)

staging_total_cents | products_total_cents
--------------------+---------------------
950                 | 950                 
(1 row)

products_total
--------------
14            
(1 row)

